# 2. Multi-Armed Bandits

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 2: Multi-armed Bandits** (Sayfa 25-46)

## İçindekiler
1. K-Armed Bandit Problemi *(s. 25-27)*
2. Action-Value Methods *(s. 27-29)*
3. Epsilon-Greedy *(s. 27-28)*
4. Upper Confidence Bound *(s. 35-36)*
5. Gradient Bandit *(s. 37-40)*

---
## 2.1 K-Armed Bandit Problemi

📖 **Referans:** Sutton & Barto, Sayfa 25-27, Section 2.1 "A k-armed Bandit Problem"

> *"Consider the following learning problem. You are faced repeatedly with a choice among k different options, or actions. After each choice you receive a numerical reward chosen from a stationary probability distribution that depends on the action you selected."* (s. 25)

### Problem Tanımı (s. 25)
- K tane slot makinesi (kol) var
- Her kolun bilinmeyen bir reward dağılımı var
- Amaç: Toplam reward'ı maksimize et

### Value of an Action (Equation 2.1, s. 25)

$$q_*(a) \doteq E[R_t | A_t = a]$$

> *"We denote the value of an arbitrary action $a$ as $q_*(a)$, which is the expected reward given that $a$ is selected."* (s. 25)

### Exploration vs Exploitation (s. 26-27)

> *"If you maintain estimates of the action values, then at any time step there is at least one action whose estimated value is greatest... If you select one of these actions, we say that you are exploiting... If instead you select one of the nongreedy actions, then we say you are exploring."* (s. 26)

In [ ]:
# Kod Örneği: K-Armed Bandit Environment
# Referans: Section 2.3 "The 10-armed Testbed" (s. 28-29)

import numpy as np
import matplotlib.pyplot as plt

class KArmedBandit:
    """
    K-Armed Bandit (10-armed testbed, s. 28-29)
    
    "The true value q*(a) of each of the ten actions was selected 
    according to a normal distribution with mean zero and unit variance."
    """
    
    def __init__(self, k=10):
        self.k = k
        self.reset()
    
    def reset(self):
        # q*(a) ~ N(0, 1) for each action (s. 28)
        self.q_true = np.random.randn(self.k)
        self.optimal_action = np.argmax(self.q_true)
        return self.q_true.copy()
    
    def step(self, action):
        """
        "The actual rewards were selected from a normal distribution 
        with mean q*(A_t) and variance 1" (s. 28)
        """
        reward = self.q_true[action] + np.random.randn()
        is_optimal = (action == self.optimal_action)
        return reward, is_optimal

# Test (Figure 2.1 benzeri, s. 28)
bandit = KArmedBandit(k=10)
print(f"True action values q*(a): {bandit.q_true.round(2)}")
print(f"Optimal action a*: {bandit.optimal_action}")

In [ ]:
# Figure 2.1 benzeri görselleştirme (s. 28)
fig, ax = plt.subplots(figsize=(10, 5))

# Her action için violin plot benzeri
for a in range(bandit.k):
    # Sample rewards
    samples = bandit.q_true[a] + np.random.randn(1000)
    
    parts = ax.violinplot([samples], positions=[a], showmeans=True)
    
    # Optimal action'ı vurgula
    if a == bandit.optimal_action:
        for pc in parts['bodies']:
            pc.set_facecolor('green')
            pc.set_alpha(0.7)

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel('Action')
ax.set_ylabel('Reward Distribution')
ax.set_title('10-Armed Testbed (Figure 2.1, s. 28)\nGreen = Optimal Action')
ax.set_xticks(range(bandit.k))
plt.show()

---
## 2.2 Action-Value Methods

📖 **Referans:** Sutton & Barto, Sayfa 27-29, Section 2.4 "Incremental Implementation"

### Sample-Average Method (Equation 2.1, s. 27)

> *"One natural way to estimate this is by averaging the rewards actually received."*

$$Q_t(a) \doteq \frac{\sum_{i=1}^{t-1} R_i \cdot \mathbb{1}_{A_i=a}}{\sum_{i=1}^{t-1} \mathbb{1}_{A_i=a}}$$

### Incremental Update (Equation 2.3, s. 31)

> *"This implementation requires memory only for $Q_n$ and $n$"* (s. 31)

$$Q_{n+1} = Q_n + \frac{1}{n}[R_n - Q_n]$$

### Genel Form (s. 31)

$$NewEstimate \leftarrow OldEstimate + StepSize[Target - OldEstimate]$$

In [ ]:
# Kod Örneği: Incremental Implementation
# Referans: Section 2.4 (s. 30-31), Equation 2.3

class BanditAgent:
    """
    Base agent with incremental update (Equation 2.3, s. 31)
    """
    
    def __init__(self, k, initial_value=0.0):
        self.k = k
        self.initial_value = initial_value
        self.reset()
    
    def reset(self):
        # Q_1(a) = initial_value (optimistic initialization, s. 34)
        self.Q = np.ones(self.k) * self.initial_value
        self.N = np.zeros(self.k)  # Action counts
    
    def select_action(self):
        raise NotImplementedError
    
    def update(self, action, reward):
        """
        Incremental update (Equation 2.3, s. 31):
        Q_{n+1} = Q_n + (1/n)[R_n - Q_n]
        """
        self.N[action] += 1
        # Step size = 1/n (sample average)
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

---
## 2.3 Epsilon-Greedy Strategy

📖 **Referans:** Sutton & Barto, Sayfa 27-28, Section 2.2 "Action-value Methods"

### Greedy Action Selection (s. 27)

> *"The greedy action selection method can be written as $A_t \doteq \arg\max_a Q_t(a)$"*

$$A_t = \arg\max_a Q_t(a)$$

### ε-Greedy (s. 28)

> *"The simplest alternative is to behave greedily most of the time, but every once in a while... select randomly from among all the actions with equal probability."*

$$A_t = \begin{cases} \arg\max_a Q_t(a) & \text{with probability } 1-\epsilon \\ \text{random action} & \text{with probability } \epsilon \end{cases}$$

> *"We call methods using this near-greedy action selection rule ε-greedy methods."* (s. 28)

In [ ]:
# Kod Örneği: ε-Greedy Agent
# Referans: Section 2.2 (s. 27-28)

class EpsilonGreedyAgent(BanditAgent):
    """
    ε-greedy action selection (s. 28)
    
    "behave greedily most of the time, but every once in a while,
    say with small probability ε, instead select randomly"
    """
    
    def __init__(self, k, epsilon=0.1, initial_value=0.0):
        super().__init__(k, initial_value)
        self.epsilon = epsilon
    
    def select_action(self):
        if np.random.random() < self.epsilon:
            # Explore: random action
            return np.random.randint(self.k)
        else:
            # Exploit: greedy action (with random tie-breaking)
            max_q = np.max(self.Q)
            max_actions = np.where(self.Q == max_q)[0]
            return np.random.choice(max_actions)

In [ ]:
# Figure 2.2 (s. 29) - ε-greedy karşılaştırma

def run_experiment(agent_class, bandit, n_steps=1000, n_runs=200, **kwargs):
    """Multiple runs over bandit problems (s. 29)"""
    
    all_rewards = np.zeros((n_runs, n_steps))
    all_optimal = np.zeros((n_runs, n_steps))
    
    for run in range(n_runs):
        bandit.reset()
        agent = agent_class(bandit.k, **kwargs)
        
        for step in range(n_steps):
            action = agent.select_action()
            reward, is_optimal = bandit.step(action)
            agent.update(action, reward)
            
            all_rewards[run, step] = reward
            all_optimal[run, step] = is_optimal
    
    return all_rewards.mean(axis=0), all_optimal.mean(axis=0)

# ε = 0, 0.01, 0.1 karşılaştırması (Figure 2.2, s. 29)
bandit = KArmedBandit(k=10)
epsilons = [0.0, 0.01, 0.1]
results = {}

for eps in epsilons:
    rewards, optimal = run_experiment(
        EpsilonGreedyAgent, bandit, n_steps=1000, n_runs=200, epsilon=eps
    )
    results[eps] = {'rewards': rewards, 'optimal': optimal}
    print(f"ε={eps}: Final avg reward = {rewards[-100:].mean():.3f}")

In [ ]:
# Figure 2.2 (s. 29) - Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'0.0': 'green', '0.01': 'red', '0.1': 'blue'}

for eps in epsilons:
    color = colors[str(eps)]
    label = f'ε={eps}' + (' (greedy)' if eps == 0 else '')
    axes[0].plot(results[eps]['rewards'], color=color, label=label)
    axes[1].plot(results[eps]['optimal'] * 100, color=color, label=label)

axes[0].set_xlabel('Steps')
axes[0].set_ylabel('Average Reward')
axes[0].set_title('Average Reward (Figure 2.2 top, s. 29)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Steps')
axes[1].set_ylabel('% Optimal Action')
axes[1].set_title('% Optimal Action (Figure 2.2 bottom, s. 29)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2.4 Upper Confidence Bound (UCB)

📖 **Referans:** Sutton & Barto, Sayfa 35-36, Section 2.7 "Upper-Confidence-Bound Action Selection"

> *"It would be better to select among the non-greedy actions according to their potential for actually being optimal, taking into account both how close their estimates are to being maximal and the uncertainties in those estimates."* (s. 35)

### UCB Action Selection (Equation 2.10, s. 35)

$$A_t \doteq \arg\max_a \left[ Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} \right]$$

> *"The square-root term is a measure of the uncertainty or variance in the estimate of a's value."* (s. 35)

- $Q_t(a)$: Exploitation term (current estimate)
- $c\sqrt{\frac{\ln t}{N_t(a)}}$: Exploration term (uncertainty bonus)
- $c$: Exploration parameter (controls exploration degree)

In [ ]:
# Kod Örneği: UCB Agent
# Referans: Section 2.7 (s. 35-36), Equation 2.10

class UCBAgent(BanditAgent):
    """
    Upper Confidence Bound action selection (Equation 2.10, s. 35)
    
    A_t = argmax_a [Q_t(a) + c * sqrt(ln(t) / N_t(a))]
    """
    
    def __init__(self, k, c=2.0, initial_value=0.0):
        super().__init__(k, initial_value)
        self.c = c  # "the number c > 0 controls the degree of exploration" (s. 35)
        self.t = 0
    
    def reset(self):
        super().reset()
        self.t = 0
    
    def select_action(self):
        self.t += 1
        
        # "actions with N_t(a) = 0 are considered to be maximizing actions" (s. 35)
        if 0 in self.N:
            return np.where(self.N == 0)[0][0]
        
        # UCB formula (Equation 2.10)
        ucb_values = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return np.argmax(ucb_values)

In [ ]:
# Figure 2.4 (s. 36) - UCB vs ε-greedy
bandit = KArmedBandit(k=10)

eps_rewards, eps_optimal = run_experiment(
    EpsilonGreedyAgent, bandit, n_steps=1000, n_runs=200, epsilon=0.1
)

ucb_rewards, ucb_optimal = run_experiment(
    UCBAgent, bandit, n_steps=1000, n_runs=200, c=2.0
)

# Plot (Figure 2.4, s. 36)
plt.figure(figsize=(10, 5))
plt.plot(eps_rewards, label='ε-greedy (ε=0.1)', color='gray')
plt.plot(ucb_rewards, label='UCB (c=2)', color='blue')
plt.xlabel('Steps')
plt.ylabel('Average Reward')
plt.title('UCB vs ε-greedy (Figure 2.4, s. 36)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 2.5 Gradient Bandit Algorithms

📖 **Referans:** Sutton & Barto, Sayfa 37-40, Section 2.8 "Gradient Bandit Algorithms"

> *"So far in this chapter we have considered methods that estimate action values and use those estimates to select actions. This is often a good approach, but it is not the only one possible."* (s. 37)

### Softmax Action Probabilities (Equation 2.11, s. 37)

$$\pi_t(a) \doteq \frac{e^{H_t(a)}}{\sum_{b=1}^{k} e^{H_t(b)}}$$

> *"Based on a numerical preference $H_t(a)$ for each action $a$... The action probabilities are determined according to a soft-max distribution."*

### Gradient Ascent Update (Equations 2.12, s. 38)

$$H_{t+1}(A_t) \doteq H_t(A_t) + \alpha (R_t - \bar{R}_t)(1 - \pi_t(A_t))$$

$$H_{t+1}(a) \doteq H_t(a) - \alpha (R_t - \bar{R}_t)\pi_t(a) \quad \forall a \neq A_t$$

> *"The $\bar{R}_t$ term serves as a baseline with which the reward is compared."* (s. 38)

In [ ]:
# Kod Örneği: Gradient Bandit
# Referans: Section 2.8 (s. 37-40), Equations 2.11-2.12

class GradientBanditAgent:
    """
    Gradient Bandit Algorithm (Section 2.8, s. 37-40)
    
    Uses preference H_t(a) instead of value estimates Q_t(a)
    """
    
    def __init__(self, k, alpha=0.1, baseline=True):
        self.k = k
        self.alpha = alpha  # "step-size parameter" (s. 38)
        self.baseline = baseline  # "use R̄_t as baseline" (s. 38)
        self.reset()
    
    def reset(self):
        # "initially all preferences are the same (e.g., all H_1(a) = 0)" (s. 37)
        self.H = np.zeros(self.k)
        self.avg_reward = 0  # R̄_t baseline
        self.t = 0
    
    def softmax(self):
        """π_t(a) = e^{H_t(a)} / Σ_b e^{H_t(b)}  (Eq. 2.11, s. 37)"""
        exp_h = np.exp(self.H - np.max(self.H))  # Numerical stability
        return exp_h / np.sum(exp_h)
    
    def select_action(self):
        probs = self.softmax()
        return np.random.choice(self.k, p=probs)
    
    def update(self, action, reward):
        """Gradient ascent update (Eq. 2.12, s. 38)"""
        self.t += 1
        probs = self.softmax()
        
        # Baseline (s. 38)
        baseline = self.avg_reward if self.baseline else 0
        
        # Update preferences (Equation 2.12)
        one_hot = np.zeros(self.k)
        one_hot[action] = 1
        
        self.H += self.alpha * (reward - baseline) * (one_hot - probs)
        
        # Update baseline: R̄_t = running average
        self.avg_reward += (reward - self.avg_reward) / self.t

In [ ]:
# Figure 2.5 (s. 39) - Effect of baseline

def run_gradient_experiment(bandit, n_steps=1000, n_runs=200, **kwargs):
    all_optimal = np.zeros((n_runs, n_steps))
    
    for run in range(n_runs):
        bandit.reset()
        # "true values q*(a) selected from N(+4, 1)" (s. 39)
        bandit.q_true += 4
        
        agent = GradientBanditAgent(bandit.k, **kwargs)
        
        for step in range(n_steps):
            action = agent.select_action()
            reward, is_optimal = bandit.step(action)
            agent.update(action, reward)
            all_optimal[run, step] = is_optimal
    
    return all_optimal.mean(axis=0)

bandit = KArmedBandit(k=10)

# With and without baseline (Figure 2.5, s. 39)
with_baseline = run_gradient_experiment(bandit, baseline=True, alpha=0.1)
without_baseline = run_gradient_experiment(bandit, baseline=False, alpha=0.1)

plt.figure(figsize=(10, 5))
plt.plot(with_baseline * 100, label='With baseline (α=0.1)', color='blue')
plt.plot(without_baseline * 100, label='Without baseline (α=0.1)', color='red')
plt.xlabel('Steps')
plt.ylabel('% Optimal Action')
plt.title('Gradient Bandit: Effect of Baseline (Figure 2.5, s. 39)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Özet

Bu notebook'ta öğrendiklerimiz (Sutton & Barto Chapter 2):

| Kavram | Sayfa | Denklem | Açıklama |
|--------|-------|---------|----------|
| Action Value | s. 25 | Eq. 2.1 | $q_*(a) = E[R_t \| A_t = a]$ |
| Sample Average | s. 27 | Eq. 2.1 | Reward'ların ortalaması |
| Incremental Update | s. 31 | Eq. 2.3 | $Q_{n+1} = Q_n + \frac{1}{n}[R_n - Q_n]$ |
| ε-greedy | s. 28 | - | Prob. ε ile random explore |
| UCB | s. 35 | Eq. 2.10 | Uncertainty bonus ile explore |
| Gradient Bandit | s. 37 | Eq. 2.11-12 | Softmax preferences |

### Yöntem Karşılaştırması (Figure 2.6, s. 42)

| Yöntem | Avantaj | Dezavantaj |
|--------|---------|------------|
| **Greedy** | Basit | Explore etmez |
| **ε-Greedy** | Basit, explore eder | Random exploration |
| **UCB** | Akıllı exploration | Parameter (c) tuning |
| **Gradient** | Soft action selection | Baseline önemli |

---
### Sonraki Notebook
**03 - Markov Decision Processes** *(Chapter 3, s. 47-72)*: Full RL problem